# Lab 04: Aggregations & Grouping

## Objectives
- Use groupBy with single and multiple aggregation functions
- Apply agg() with count, sum, avg, min, max
- Use window functions for running totals and rankings
- Compare PySpark aggregations with Spark SQL equivalents

## Exam Domains
- **Developing DataFrame/DataSet API Applications — 30%**
- **Using Spark SQL — 20%**

## Estimates
- **Time:** ~35 minutes
- **Cost:** ~$1 (serverless)
- **Compute:** Serverless

![Aggregation & Join Flow](../assets/diagrams/lab-04-aggregation-join-flow.png)

In [ ]:
USE_CATALOG = "spark_lab_guide"
USE_SCHEMA = "default"

spark.sql(f"USE CATALOG {USE_CATALOG}")
spark.sql(f"USE SCHEMA {USE_SCHEMA}")

taxi_df = spark.table("taxi_trips")
github_df = spark.table("github_events")

## A. Basic groupBy + Single Aggregation

`groupBy()` groups rows by one or more columns. Follow it with an aggregation function to compute a summary per group.

In [ ]:
from pyspark.sql.functions import count, sum, avg, min, max, round, col

# Count trips per pickup location
trips_by_location = (
    taxi_df
    .groupBy("PULocationID")
    .count()
    .orderBy(col("count").desc())
)
trips_by_location.show(10)

## B. Multiple Aggregations with agg()

`agg()` lets you compute multiple aggregations in a single pass — more efficient than calling separate aggregations.

In [ ]:
# Multiple aggregations per group
location_stats = (
    taxi_df
    .groupBy("PULocationID")
    .agg(
        count("*").alias("trip_count"),
        round(avg("total_amount"), 2).alias("avg_fare"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        max("tip_amount").alias("max_tip"),
    )
    .orderBy(col("total_revenue").desc())
)
location_stats.show(10)

### SQL Alternative

In [ ]:
taxi_df.createOrReplaceTempView("taxi")

spark.sql("""
    SELECT
        PULocationID,
        COUNT(*) AS trip_count,
        ROUND(AVG(total_amount), 2) AS avg_fare,
        ROUND(SUM(total_amount), 2) AS total_revenue,
        ROUND(AVG(trip_distance), 2) AS avg_distance,
        MAX(tip_amount) AS max_tip
    FROM taxi
    GROUP BY PULocationID
    ORDER BY total_revenue DESC
    LIMIT 10
""").show()

## C. Grouping by Multiple Columns

You can group by multiple columns to create finer-grained aggregations.

In [ ]:
# Group by pickup location AND payment type
payment_by_location = (
    taxi_df
    .groupBy("PULocationID", "payment_type")
    .agg(
        count("*").alias("trip_count"),
        round(avg("total_amount"), 2).alias("avg_fare"),
    )
    .orderBy(col("trip_count").desc())
)
payment_by_location.show(10)

## D. Aggregations on GitHub Events

Let's apply the same patterns to the GitHub Archive data to see event type distributions.

In [ ]:
# Count events by type
event_counts = (
    github_df
    .groupBy("type")
    .agg(count("*").alias("event_count"))
    .orderBy(col("event_count").desc())
)
event_counts.show()

In [ ]:
# Top contributors by number of events
top_actors = (
    github_df
    .groupBy(col("actor.login").alias("username"))
    .agg(count("*").alias("event_count"))
    .orderBy(col("event_count").desc())
)
top_actors.show(10)

## E. Window Functions

Window functions compute values across a set of rows related to the current row — without collapsing rows like `groupBy`. Useful for rankings, running totals, and row comparisons.

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, lag

# Rank pickup locations by total revenue
window_spec = Window.orderBy(col("total_revenue").desc())

ranked_locations = (
    location_stats
    .withColumn("rank", rank().over(window_spec))
    .withColumn("dense_rank", dense_rank().over(window_spec))
    .withColumn("row_number", row_number().over(window_spec))
)
ranked_locations.select("PULocationID", "total_revenue", "rank", "dense_rank", "row_number").show(10)

In [ ]:
# Running total with window — partitioned by payment_type
window_cumulative = (
    Window
    .partitionBy("payment_type")
    .orderBy("PULocationID")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

cumulative_revenue = (
    payment_by_location
    .withColumn("cumulative_trips", sum("trip_count").over(window_cumulative))
)
cumulative_revenue.filter(col("payment_type") == 1).show(10)

### SQL Alternative — Window Functions

In [ ]:
spark.sql("""
    SELECT
        PULocationID,
        total_amount,
        RANK() OVER (ORDER BY total_amount DESC) as rank,
        SUM(total_amount) OVER (
            ORDER BY tpep_pickup_datetime
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) as running_total
    FROM taxi
    LIMIT 10
""").show()

> **Exam Tip:** Know the difference between `rank()`, `dense_rank()`, and `row_number()`:
> - `rank()` — ties get the same rank, next rank is skipped (1, 2, 2, 4)
> - `dense_rank()` — ties get the same rank, next rank is NOT skipped (1, 2, 2, 3)
> - `row_number()` — no ties, every row gets a unique number (1, 2, 3, 4)

## Exam-Style Review

**Q1.** What is the advantage of using `agg()` with multiple functions over chaining separate aggregation calls?
- A) `agg()` is lazily evaluated, separate calls are eager
- B) `agg()` computes all aggregations in a single pass over the data
- C) `agg()` supports more aggregation functions
- D) There is no difference

**Q2.** What does `rank()` do when two rows have the same value?
- A) Assigns random ranks
- B) Assigns the same rank and skips the next rank number
- C) Assigns the same rank and continues with the next sequential number
- D) Throws an error

**Q3.** What does `Window.partitionBy("col")` do?
- A) Physically repartitions the data across the cluster
- B) Defines the group within which the window function operates
- C) Filters rows by the column value
- D) Sorts the data by the column

**Q4.** What SQL clause is equivalent to `groupBy("col").agg(count("*"))`?
- A) `PARTITION BY col`
- B) `GROUP BY col` with `COUNT(*)`
- C) `ORDER BY col`
- D) `WHERE col IS NOT NULL`

### Answers
- **Q1: B** — `agg()` processes all aggregations in a single pass, which is more efficient.
- **Q2: B** — `rank()` assigns the same rank to ties and skips the next number. `dense_rank()` does not skip.
- **Q3: B** — `partitionBy` in a Window defines the group (partition) within which the function computes, not a physical repartition.
- **Q4: B** — `GROUP BY col` with `COUNT(*)` is the SQL equivalent.

## Key Takeaways
- `groupBy().agg()` is the standard pattern for multiple aggregations in one pass
- Window functions (`rank`, `dense_rank`, `row_number`, `sum().over()`) compute across related rows without collapsing them
- `Window.partitionBy()` defines the group; `Window.orderBy()` defines the order within that group
- All PySpark aggregations have Spark SQL equivalents (`GROUP BY`, `OVER()`)

**Next:** [Lab 05 — Joins & Set Operations](05-joins-set-operations.ipynb)

In [ ]:
spark.catalog.dropTempView("taxi")
print("Lab 04 cleanup complete.")